# 2-1-4 기계학습 유형과 알고리즘 선정
교과서 **74~75쪽**

지난 시간에는 표를 손질했습니다.  
오늘은 **어떤 종류의 학습인지** 고르고, **k-최근접 이웃**이 새 꽃을 어떻게 분류하는지 눈으로 확인합니다.

| 오늘 할 일 | 왜 필요한가? |
|---|---|
| 지도/비지도/강화를 구분하고, k-NN으로 새 꽃 한 송이를 분류해 보기 | 문제에 맞는 유형·알고리즘을 고르지 않으면 학습이 엇나갑니다 |

**준비물:** 같은 폴더의 `Iris1.csv`  
**실행:** 위에서부터 ▶.


## 0. 도구 꺼내기 · 파일 열기


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import platform

if platform.system() == 'Windows':
    plt.rc('font', family='Malgun Gothic')
elif platform.system() == 'Darwin':
    plt.rc('font', family='AppleGothic')
plt.rcParams['axes.unicode_minus'] = False

df = pd.read_csv('Iris1.csv', encoding='cp949')
print('크기:', df.shape, '/ 종류:', df['종류'].unique().tolist())
df.head()


---

# 1. 기계학습 유형은 어떻게 나눌까? (74쪽)

나누는 기준은 **정답(레이블)을 주고 배우느냐**입니다. 붓꽃 표에서 레이블은 `종류`입니다.

| 유형 | 정답 | 하는 일 | 붓꽃 |
|---|---|---|---|
| **지도학습 · 분류** | 있음 (갈래) | 몇 갈래 중 하나를 맞힘 | (나) 새 꽃의 종류 |
| **지도학습 · 예측(회귀)** | 있음 (숫자) | 연속된 숫자를 맞힘 | (다) 빠진 꽃받침 너비 |
| **비지도학습 · 군집** | 없음 | 비슷한 것끼리 묶음 | (가) 숫자만 보고 무리 나누기 |
| **강화 학습** | 표가 아니라 보상·벌 | 해 보며 방법을 고침 | 오늘은 쓰지 않음 (로봇·게임) |


In [ ]:
# 지도학습: 입력 X 와 정답 y 를 나눕니다.
X = df[['꽃받침 길이', '꽃받침 너비', '꽃잎 길이', '꽃잎 너비']]
y = df['종류']
print('X:', X.shape, '/ y:', y.shape)
print(y.value_counts())


In [ ]:
# 비지도학습: 정답 열을 빼면 이름표 없는 숫자 표가 됩니다.
군집용 = df[['꽃받침 길이', '꽃받침 너비', '꽃잎 길이', '꽃잎 너비']]
print('정답을 뺀 표:', 군집용.shape)
군집용.head()


> **생각하기 1**
>
> | 상황 | 지도·분류 / 지도·예측 / 비지도 / 강화 |
> |---|---|
> | (가) 종류 열 없이 숫자만 보고 꽃을 몇 무리로 나누기 |  |
> | (나) 종류를 아는 꽃으로 배운 뒤 새 꽃의 종류 맞히기 |  |
> | (다) 꽃잎 길이·너비로 꽃받침 너비라는 **숫자** 구하기 |  |
> | 로봇 청소기가 부딪히면 벌, 방을 다 쓸면 칭찬 |  |
> →


---

# 2. k-최근접 이웃으로 새 꽃 분류하기 (75쪽)

가까운 k송이의 **다수결**로 종류를 정합니다.  
교과서 그림 Ⅱ-4는 k = 5, 주변에 버시컬러 4 + 버지니카 1 → **버시컬러**입니다.

오늘은 그림이 잘 보이게 **꽃잎 길이·너비**만 씁니다. 거리는 피타고라스입니다.


In [ ]:
새꽃_길이, 새꽃_너비 = 4.9, 1.6   # 교과서 그림과 같은 상황

연습 = df[['꽃잎 길이', '꽃잎 너비', '종류']].copy()
연습['거리'] = ((연습['꽃잎 길이'] - 새꽃_길이) ** 2 + (연습['꽃잎 너비'] - 새꽃_너비) ** 2) ** 0.5
연습.sort_values('거리').head(10)


In [ ]:
k = 5
이웃 = 연습.nsmallest(k, '거리')
print('가장 가까운', k, '송이')
display(이웃)
print(이웃['종류'].value_counts())
print('다수결:', 이웃['종류'].value_counts().idxmax())


> **생각하기 2**
> k = 5 일 때 이 새 꽃은 어떤 종류인가요? 이웃이 4 대 1이면 왜 그 쪽으로 결정하나요?
> →


In [ ]:
색 = {'세토사': 'tab:red', '버시컬러': 'tab:orange', '버지니카': 'tab:green'}
plt.figure(figsize=(6.5, 4.5))
for 이름, 부분 in 연습.groupby('종류'):
    plt.scatter(부분['꽃잎 길이'], 부분['꽃잎 너비'], c=색[이름], label=이름, alpha=0.55, s=36)
plt.scatter(이웃['꽃잎 길이'], 이웃['꽃잎 너비'],
            facecolors='none', edgecolors='black', s=140, linewidths=2, label='가까운 5송이')
plt.scatter([새꽃_길이], [새꽃_너비], marker='*', c='black', s=260, label='새 꽃', zorder=5)
plt.xlabel('꽃잎 길이'); plt.ylabel('꽃잎 너비')
plt.title('k = 5 최근접 이웃'); plt.legend(); plt.grid(alpha=0.3)
plt.show()


k는 사람이 정하는 값(하이퍼파라미터)입니다. 같은 새 꽃에서 k만 바꿔 봅시다.


In [ ]:
def k개로_분류(k값):
    투표 = 연습.nsmallest(k값, '거리')['종류'].value_counts()
    return 투표.idxmax(), 투표.to_dict()

print('새 꽃 (4.9, 1.6)')
for k값 in [1, 5, 15]:
    결과, 투표 = k개로_분류(k값)
    print(f'k = {k값:2}  →  {결과}   {투표}')


경계 근처로 옮기면 k에 따라 답이 달라질 수 있습니다. 숫자를 바꿔 다시 실행해 보세요.


In [ ]:
새꽃_길이, 새꽃_너비 = 5.0, 1.7
연습['거리'] = ((연습['꽃잎 길이'] - 새꽃_길이) ** 2 + (연습['꽃잎 너비'] - 새꽃_너비) ** 2) ** 0.5

print('새 꽃 (5.0, 1.7)')
for k값 in [1, 5, 15]:
    결과, 투표 = k개로_분류(k값)
    print(f'k = {k값:2}  →  {결과}   {투표}')


> **생각하기 3**
> 1. k = 1 인데 가장 가까운 한 송이가 측정 실수라면?
> →
> 2. (5.0, 1.7)에서 k = 1과 k = 5의 결과가 같았나요, 달랐나요?
> →


> **생각하기 4**
> (나) 종류 맞히기는 왜 **지도학습 · 분류 · k-NN** 후보인가요? 한 줄로 적으세요.
> →


---

# 오늘 정리

| 확인 항목 | 내가 본 결과 |
|---|---|
| (가)(나)(다)의 유형 |  |
| k = 5 다수결 결과 |  |
| k를 바꾸면 |  |

**자기 점검**

- [ ] 지도 / 비지도 / 강화, 분류 / 예측을 구분할 수 있다.
- [ ] k-NN이 가까운 k개의 다수결임을 설명할 수 있다.
- [ ] k = 5 일 때 그림 Ⅱ-4의 새 점이 버시컬러가 되는 이유를 말할 수 있다.

다음 시간에는 `KNeighborsClassifier`로 모델을 만들고 점수를 매깁니다.
